In [2]:
import gzip
import re
import folium
from folium.plugins import Fullscreen
import geopandas as gpd
import pandas as pd

def nettoyer_texte(texte):
  if not isinstance(texte, str):
    return ""
  return (
      texte.replace("–", "-")
      .replace("—", "-")
      .replace("\u2013", "-")
      .replace("\u2014", "-")
      .strip()
  )


print("Chargement du fichier Excel...")
df_excel = pd.read_excel("Liste_PAPI (1).xlsx")
df_excel["Projet_clean"] = (
    df_excel["Projet"]
    .astype(str)
    .apply(nettoyer_texte)
    .str.replace(r"\s+", " ", regex=True)
)

projets_pep = [
    "PEP Paladru-Fure-Morge-Olon-Roize (2020-2024)",
    "PEP des 4 Vallées (2019-2023)",
    "PEP de l'Ain aval et de ses affluents (2022-2026)",
    "PEP du bassin de l'Allan (2025-2027)",
    "PEP Baie de Saint-Brieuc Ouest (2025-2026)",
    "PEP Rivière de Morlaix (2025-2027)",
    "PEP de Lorient Agglomération (2024-2027)",
    "PEP de Cap Atlantique (2024-2026)",
    "PEP Côtes des Légendes (2025-2027)",
    "Projet de PEP de la Presqu'île de Crozon (2025-2027)",
    "PEP Fleuves de la Baie du Mont-Saint-Michel (2022-2032)",
    "PEP des Vals du Blésois (2025-2028)",
    "PEP Juine-Essonne-École (2025-2027)",
    "PEP Cher median et aval (2024-2026)",
    "PEP Strasbourg Confluences (2023-2025)",
    "PEP Châlons-en-Champagne et Marne Moyenne (2024-2026)",
    "PEP Giessen-Lièpvrette (2021-2025)",
    "PEP Souffel (2024-2026)",
    "PEP des deux Morin (2023-2025)",
    "PEP des Basses vallées de l'Orne et de la Dives (2025-2027)",
    "PEP du Cirieux (2025–2026)",
    "PEP de l'Iton (2024-2026)",
    "PEP Bassin d’Arcachon (2023-2026)",
    "PEP Brouage (2025-2027)",
    "PEP Val-de-Garonne Agglomération (2025-2027)",
    "PEP Fleuves de la baie du Mont-Saint-Michel (2024-2026)",
    "PEP Garonne en Gironde (2024-2026)",
    "PEP Côtier Basque (2025–2027)",
    "PEP Polders et Marais de la Baie du Mont-Saint-Michel (2024-2026)",
    "PEP Agen (2026-2028)",
    "PEP Romanche (2022-2026)",
    "PEP Tech-Albères (2021-2026)",
    "PEP Tarn aval 12-81-31 (2025-2027)",
    "PEP du Réart (2024-2026)",
    "PEP Moselle Amont (2023-2026)",
    "PEP du Grand Hers (2024-2026)",
    "PEP du Gave de Pau Bigourdan (2022-2026)",
    "PEP Garonne Amont (2023-2026)",
    "PEP Dordogne Lotoise (2024–2026)",
    "PEP Aveyron Amont (2024-2026)",
    "PEP du Bassin Versant de la Siagne (2021-2026)",
    "PEP Verdon (2024-2028)",
    "PEP Nive (2026-2032)",
    "PEP Gave d’Oloron soumis à Evaluation Environnementale (2025–2027)",
    "PEP de la Sorgue et du réseau de ses bras (2025-2027)",
    "PEP de la Mayenne et de l'Oudon (2023-2026)",
    "PEP de l'Estuaire de la Loire (Rive Sud / Pays de Retz) (2021-2024)",
    "PEP du Loir (2022-2025)",
]

projets_premiers_papi = [
    "PAPI de la Sanne et du Dolon (2024-2026)",
    "PAPI du Lez Rhodanien (2021-2027)",
    "PAPI Ozon (2019-2026)",
    "PAPI Cance-Deûme (2023-2028)",
    "PAPI du Puy-en-Velay (2025-2030)",
    "PAPI de Riom (2025-2029)",
    "PAPI Clermont Auvergne Métropole (2025-2031)",
    "PAPI des Rivières du Beaujolais (2022-2027)",
    "PAPI des bassins de la Tille, de la Vouge et de l'Ouche (2025-2031)",
    "PAPI de Saint-Malo (2018-2026)",
    "PAPI de l'Odet (2024-2030)",
    "PAPI de la Communauté d’Agglomération de Bastia (2023-2027)",
    "PAPI de la Communauté de Communes de Marana-Golo (2023-2027)",
    "PAPI Moselle Aval (2026-2032)",
    "PAPI Sarre (2026-2032)",
    "PAPI Moder (2026-2032)",
    "PAPI Nord Alsace (2026-2032)",
    "PAPI de la Marne amont et de ses affluents (2024-2029)",
    "PAPI Bruche-Mossig (2026-2032)",
    "PAPI Doller (2026-2032)",
    "PAPI de l'Ill Amont (2025-2031)",
    "PAPI de la Sambre (2026-2031)",
    "PAPI Le Havre Estuaire Pointe de Caux (2025-2031)",
    "PAPI des ruisseaux couverts (2023-2028)",
    "PAPI des bassins du Cérou et de la Vère (2025-2030)",
    "PAPI du bassin de la Sarthe (2026-2032)",
]

projets_papi_matures = [
    "PAPI Ardeche (2023-2026)",
    "PAPI Val de Saône Franc-Lyonnais (2026-2031)",
    "PAPI Yzeron (2026-2032)",
    "PAPI Arve (2020-2026)",
    "PAPI du Pays de Maurienne (2025-2031)",
    "PAPI du Lot (2026-2032)",
    "PAPI de la Verse (2021-2026)",
    "PAPI de la Communauté d'Agglomération du Pays Ajaccien (2024-2028)",
    "PEP Meurthe (2026-2032)",
    "PAPI Arcques et Scie (2022-2027)",
    "PEP Montluçon Cher amont (2024-2026)",
    "PAPI de l'Ondaine (2024-2026)",
    "PAPI de la Joyeuse (2026-2031)",
    "PAPI Veore Barberolle (2026-2031)",
    "PAPI 2 du Garon (2028-2033)",
    "PAPI 2 du Gier (2026-2032)",
    "PAPI de la Bourbre (2020-2026)",
    "PAPI de Nevers (2020-2026)",
    "PAPI Brévenne-Turdine (2021-2027)",
    "PAPI de la Combe de Savoie (2014-2022)",
    "PAPI des Olonnes (2023-2027)",
    "PAPI Gave de Pau (2026-2031)",
    "PAPI Isère Amont (2012-2022)",
    "PAPI Bresle-Somme-Authie (2023-2028)",
    "PAPI de l'Azergues (2019-2027)",
    "PAPI Val d'Ariège (2026-2031)",
    "PAPI du Boulonnais (2025-2031)",
    "PAPI Valloire-Galaure (2017-2026)",
    "PAPI de l'Herbasse (2016-2026)",
    "PAPI Veaune, Bouterne, Torras et petits affluents du Rhône (2019-2026)",
    "PAPI Ouveze (2024-2029)",
    "PAPI Beaujolais (2021-2026)",
    "PAPI Lac du Bourget (2021_2026)",
    "PAPI Tarentaise (2023-2029)",
    "PAPI du Furan (2023-2029)",
    "PAPI de Troyes et du Bassin de la Seine supérieure (2020-2026)",
    "PAPI des Vals de la Loire dans le Cher, la Nièvre et le Giennois (2021-2026)",
    "PAPI du Loing (2025-2031)",
    "PAPI de Yerres (2026-2031)",
    "PAPI Val de Saône et côte viticole (2025-2031)",
    "PAPI du bassin de l'Yonne (2025-2031)",
    "PAPI du secteur de Besançon (2015-2022)",
    "PAPI 2 Blavet (2022-2028)",
    "PAPI 2 Ellé-Isole-Laïta (2022-2028)",
    "PAPI Bruilhois (2024-2030)",
    "PAPI 2 Arguenon (2024-2029)",
    "PAPI 2 Littoral Sud Finistère (2024-2030)",
    "PAPI 2 de Golfe du Morbihan Vannes Agglomération (GMVA) (2026-2032)",
    "PEP Aulne 2 (2025-2028)",
    "PAPI des Vals de l'Orléanais (2026-2032)",
    "PAPI Interrégional du Loing (2026-2031)",
    "PAPI des Vals de Loire dans la Nièvre, le Cher et le Giennois (2023-2029)",
    "PAPI de Tours Métropole Val de Loire (2020–2027)",
    "PAPI Vienne-Clain (2023-2029)",
    "PAPI Meuse 3 (2025-2030)",
    "PAPI 3 Ried Centre Alsace (IRCA) (2026-2032)",
    "PAPI Troyes et Bassin de la Seine Supérieure (2020-2026)",
    "PAPI Serein-Armançon (2027-2032)",
    "PAPI de la Somme (2021-2027)",
    "PAPI Canche (2022-2027)",
    "PAPI Lys (2021-2027)",
    "PAPI Andomarois (2020-2025)",
    "PAPI du Delta de l'Aa (2016 – 2024)",
    "PAPI de la Vallée de l'Oise (2026-2032)",
    "PAPI Seine et Marne franciliennes n°2 (2023-2029)",
    "PAPI du Loing (2025-2031)",
    "PAPI Vallée de l'Oise (2026-2032)",
    "PAPI du bassin de l'Yonne (2026-2032)",
    "PAPI Rouen-Louviers-Austreberthe (2024-2030)",
    "PAPI Côte des Havres (2021-2024)",
    "PAPI Bresle–Somme–Authie (2025-2030)",
    "PEP du Bassin de la Sarthe (2023-2026)",
    "PAPI Dax (2020-2026)",
    "PAPI Estuaire Charente (2024-2030)",
    "PAPI Estuaire Gironde (2025-2031)",
    "PAPI Fleuve Charente (2024-2030)",
    "PAPI Île de Ré 2 (2021-2024)",
    "PAPI La Rochelle (2013-2018)",
    "PAPI Nord-Aunis (2014–2026)",
    "PAPI Oléron (2021-2024)",
    "PAPI Seudre (2021-2024)",
    "PAPI Aix, Fouras, Yves, Châtelaillon (2017–2022)",
    "PAPI Dordogne 2 (2015-2019)",
    "PAPI Vienne Aval (2023-2029)",
    "PAPI Nivelle (2015-2021)",
    "PAPI Vendée Sèvres Autize (2017-2023)",
    "PEP Agglomération rochelaise (2025–2027)",
    "PEP Adour Aval soumis à Evaluation Environnementale (2025-2027)",
    "PEP Dordogne 2 (2025-2027)",
    "PEP Oléron (2025-2027)",
    "PAPI 3 Vistre (2022-2028)",
    "PAPI 3 Vidourle (2024-2029)",
    "PAPI de Toulouse (2026-2031)",
    "PAPI de Thau (2026-2031)",
    "PAPI 3 de la Têt (2024-2029)",
    "PAPI Tarn-Sorgues-Dourdou-Rance (2023-2028)",
    "PAPI Tarn aval 82 (2027-2032)",
    "PAPI Tarn-amont (2024-2029)",
    "PAPI Salat et Volp (2024-2030)",
    "PAPI Orb-Libron (2024-2029)",
    "PAPI 3 du bassin de l'Or (2026-2032)",
    "PAPI Neste 2 (2024-2030)",
    "PAPI Montauban-Moissac (2027-2032)",
    "PAPI Lèze (2024-2030)",
    "PAPI 3 Lez (2022-2028)",
    "PAPI 3 Gardons (2021-2027)",
    "PAPI 3 du fleuve Hérault (2023-2028)",
    "PAPI 3 du Gard Rhodanien (2024-2030)",
    "PAPI Cèze et petits affluents du Rhône (2023-2028)",
    "PAPI 3 du Bassin de l'Aude et de la Berre (2023-2028)",
    "PAPI de l'Agout (2025-2030)",
    "PAPI Agly (2023-2028)",
    "PEP Adour Amont (2020-2026)",
    "PAPI du Var (2023-2028)",
    "PAPI des Paillons (2014-2021)",
    "PEP Maralpin (2017-2022)",
    "PAPI Durance (2024-2030)",
    "PAPI de l’Argens et des côtiers de l’Estérel (2016-2025)",
    "PAPI des bassins versants de l'Huveaune et des Aygalades (2021-2026)",
    "PAPI de l’Ouvèze Provençale (2025-2031)",
    "PAPI des bassins versants du Loup, de la Brague et des vallons côtiers (2014-2021)",
    "PAPI Cagne-Malvan (2016-2022)",
    "PAPI 2 Calavon-Coulon (2024-2030)",
    "PAPI du bassin versant du Gapeau (2021-2026)",
    "PAPI du Rhône Aval et de la Camargue (2022-2028)",
    (
        "Arc, Cadière, Touloubre et affluents de l'Étang de Berre"
        " (2024-2030)"
    ),
    "PEP Durance Amont et Hautes-Vallées Alpines (2024-2027)",
    "PAPI Vilaine 3 (2020-2027)",
    "PAPI Estuaire de la Loire et Agglomération Nantaise (2022-2028)",
    "PAPI Basses Vallées Angevines (2024-2030)",
    "PAPI des Vals d'Authion et de la Loire (2022-2028)",
    "PAPI du Bassin de la Vendée (2012-2023)",
    "PAPI de la Sèvre Nantaise (2023-2025)",
    "PAPI du Pays de Saint-Gilles-Croix-de-Vie (2015-2022)",
    "PAPI de la Presqu'île Guérandaise (2017-2023)",
    "PAPI du Bassin du Lay aval (2014-2024)",
    "PAPI de la Baie de Bourgneuf (2014-2024)",
    "PAPI de la Carene (2021-2026)",
    "PAPI Drac (2026-2032)",
    "PAPI Orge-Yvette (2026-2031)",
    "PAPI Haute Zorn (2026-2032)",
    "PAPI Zorn Aval et Landgraben (2026-2032)",
]

def extraire_insee(liste_projets):
  liste_projets_propres = [nettoyer_texte(p) for p in liste_projets]
  df_f = df_excel[df_excel["Projet_clean"].isin(liste_projets_propres)]
  mapping = {}
  for _, row in df_f.iterrows():
    nom_projet = row["Projet_clean"]
    texte_communes = str(row["Communes couvertes"])
    codes = re.findall(r"\((\d{5})\)", texte_communes)
    for code in codes:
      mapping[code] = nom_projet
  return mapping


insee_pep = extraire_insee(projets_pep)
insee_papi_1 = extraire_insee(projets_premiers_papi)
insee_papi_matures = extraire_insee(projets_papi_matures)
print("Chargement du fichier géographique...")
with gzip.open("communes-100m.geojson.gz", "rb") as f:
  gdf_communes = gpd.read_file(f)

if gdf_communes.crs is not None:
  gdf_communes = gdf_communes.to_crs(epsg=4326)

colonne_code = "code"
for col in ["code", "INSEE_COM", "insee", "code_insee"]:
  if col in gdf_communes.columns:
    colonne_code = col
    break

colonne_nom = "nom"
for col in ["nom", "NOM", "nom_commune", "NOM_COMM", "libelle", "LIBELLE"]:
  if col in gdf_communes.columns:
    colonne_nom = col
    break
      
gdf_pep = gdf_communes[
    gdf_communes[colonne_code].astype(str).isin(insee_pep.keys())
].copy()
gdf_pep["Projet_PEP"] = gdf_pep[colonne_code].astype(str).map(insee_pep)

gdf_papi_1 = gdf_communes[
    gdf_communes[colonne_code].astype(str).isin(insee_papi_1.keys())
].copy()
gdf_papi_1["Projet_PEP"] = gdf_papi_1[colonne_code].astype(str).map(insee_papi_1)

gdf_papi_mat = gdf_communes[
    gdf_communes[colonne_code].astype(str).isin(insee_papi_matures.keys())
].copy()
gdf_papi_mat["Projet_PEP"] = (
    gdf_papi_mat[colonne_code].astype(str).map(insee_papi_matures)
)

def style_pep(feature):
  return {
      "fillColor": "#e6550d",
      "fillOpacity": 0.75,
      "color": "#FFFFFF",
      "weight": 0.5,
  }


def style_papi_1(feature):
  return {
      "fillColor": "#1f77b4",
      "fillOpacity": 0.75,
      "color": "#FFFFFF",
      "weight": 0.5,
  }


def style_papi_mat(feature):
  return {
      "fillColor": "#2ca02c",
      "fillOpacity": 0.75,
      "color": "#FFFFFF",
      "weight": 0.5,
  }

print("Génération de la carte...")
m = folium.Map(tiles="OpenStreetMap")

Fullscreen(
    position="topleft",
    title="Plein écran",
    title_cancel="Quitter",
    force_separate_button=True,
).add_to(m)

# 1. Couche PEP (Orange)
folium.GeoJson(
    gdf_pep,
    name="Projets PEP (Orange)",
    style_function=style_pep,
    tooltip=folium.JsonTooltip
    if hasattr(folium, "JsonTooltip")
    else folium.GeoJsonTooltip(
        fields=[colonne_nom, colonne_code, "Projet_PEP"],
        aliases=["Commune :", "Code INSEE :", "Projet PEP :"],
        localize=True,
    ),
    popup=folium.GeoJsonPopup(
        fields=[colonne_nom, colonne_code, "Projet_PEP"],
        aliases=["Nom :", "Code INSEE :", "Projet PEP :"],
        localize=True,
        labels=True,
    ),
).add_to(m)

folium.GeoJson(
    gdf_papi_1,
    name="Premiers PAPI (Bleu)",
    style_function=style_papi_1,
    tooltip=folium.GeoJsonTooltip(
        fields=[colonne_nom, colonne_code, "Projet_PEP"],
        aliases=["Commune :", "Code INSEE :", "Projet PAPI :"],
        localize=True,
    ),
    popup=folium.GeoJsonPopup(
        fields=[colonne_nom, colonne_code, "Projet_PEP"],
        aliases=["Nom :", "Code INSEE :", "Projet PAPI :"],
        localize=True,
        labels=True,
    ),
).add_to(m)

folium.GeoJson(
    gdf_papi_mat,
    name="PAPI Matures / Renouvelés (Vert)",
    style_function=style_papi_mat,
    tooltip=folium.GeoJsonTooltip(
        fields=[colonne_nom, colonne_code, "Projet_PEP"],
        aliases=["Commune :", "Code INSEE :", "Projet PAPI :"],
        localize=True,
    ),
    popup=folium.GeoJsonPopup(
        fields=[colonne_nom, colonne_code, "Projet_PEP"],
        aliases=["Nom :", "Code INSEE :", "Projet PAPI :"],
        localize=True,
        labels=True,
    ),
).add_to(m)

folium.LayerControl(position="topright", collapsed=False).add_to(m)

toutes_geometries = pd.concat([gdf_pep, gdf_papi_1, gdf_papi_mat])
if not toutes_geometries.empty:
  minx, miny, maxx, maxy = toutes_geometries.total_bounds
  m.fit_bounds([[miny, minx], [maxy, maxx]])

m

Chargement du fichier Excel...
Chargement du fichier géographique...
Génération de la carte...
